# PRÁCTICA 8.1: CRAWLER

## PLANTEAMIENTO DE LA PRÁCTICA

Dado que nuestro objetivo es crear un recomendador de universidades para los alumnos, se ha decidido extraer la información que puede ser relevante para ellos a la hora de seleccionar una universidad:


1.- **Nombre** de la universidad.

2.- **Tipo de universidad**, es decir, si es pública o privada.

3.- Su **fecha de fundación**, que permite saber al alumno si es una universidad con tradición o una universidad joven, de reciente formación.

4.- El **presupuesto** de la universidad, que permita conocer el dinero disponible para proyectos de investigación, innovación ,etc. Esto puede
    ser interesante, sobre todo, de cara a los alumnos de doctorado o a personal de investigación que esté pensando en trasladarse a la 
    universidad.
    
5.- El **Rector** (o Presidente) de la universidad. Esto permite al alumno investigar sobre la carrera de la persona de máxima autoridad en la
    universidad.
    
6.- El **número de estudiantes**, que le permite al alumno tener una idea del tamaño de la universidad.

7.- La **dirección web** de la Universidad en caso de que el alumno quisiera obtener más información acerca de la universidad.

Otros campos tales como el lema de la universidad, los colores o puestos por debajo del rector (vice-rector), etc. se han considerado datos no esenciales y que provocarían una sobrecarga de información que alejaría al alumno de la información no esencial. 



## Consideración previa

Partimos de los datos generados en la práctica anterior, es decir, considero que ya disponemos de la carpeta que contiene los htmls de las 
universidades de Alemania (university_htmls). (Dado que el aula virtual me lo permitía por tamaño, incluyo la carpeta "university_htmls" en la entrega de la práctica).

In [31]:
import os
import csv
from bs4 import BeautifulSoup


In [33]:
def extract_field_value(table, label):
    """
    Extrae el valor de un campo específico de una tabla HTML, buscando por el texto del encabezado. Esta función es general y la usaremos 
    para extraer todos los campos que nos interesan.

    Input:
        table (Tag): La tabla HTML que contiene los datos.
        label (str): El texto del encabezado que queremos buscar.

    Output:
        str: El valor del campo correspondiente o "No applicable" si no se encuentra.

    Ejemplo:
        Entrada:
        - label: "Type"
        - tabla: Contiene una fila con `<th>Type</th><td>Public</td>`
        Salida: "Public"
    """
    # Buscamos la celda <th> con el encabezado especificado por 'label'.
    # 
    # Usamos `table.find()` para buscar dentro de la tabla HTML la primera celda <th> cuyo texto coincida exactamente
    # con el valor de `label` (por ejemplo, "President"). Esto nos permite localizar la fila de la tabla que contiene
    # la información correspondiente a este campo.
    #
    # Ejemplo de HTML:
    # <tr>
    #   <th scope="row" class="infobox-label" style="padding-right:0.65em;">
    #       <a href="/wiki/University_president" class="mw-redirect" title="University president">President</a>
    #   </th>
    #   <td class="infobox-data">
    #       <a href="/w/index.php?title=Geraldine_Rauch&amp;action=edit&amp;redlink=1" class="new" title="Geraldine Rauch (page does not exist)">
    #           Geraldine Rauch
    #       </a>
    #       <small>(since 2022)</small>
    #   </td>
    # </tr>
    #
    # Lo que sucedería sería entonces:
    # - Si `label = "President"`, `table.find("th", string=label)` encuentra esta celda:
    #   <th scope="row" class="infobox-label" style="padding-right:0.65em;">
    #       <a href="/wiki/University_president" class="mw-redirect" title="University president">President</a>
    #   </th>
    row = table.find("th", string=label)

    # Si encontramos la celda <th>, buscamos la celda <td> correspondiente.
    # 
    # ¿Qué hace esta condición?
    # Verificamos si `row` tiene un valor válido, es decir, si realmente encontramos la celda <th> que contiene
    # el encabezado especificado por `label` (por ejemplo, "President").
    # Si `row` es None, significa que el campo no existe en la tabla y no necesitamos continuar.
    if row:
        
        # Buscamos la celda <td> que sigue a la celda <th> que encontramos.
        # 
        # 
        # Usamos `find_next_sibling("td")` para localizar la celda <td> que está inmediatamente después del <th>.
        # Esta celda <td> es donde suele almacenarse la información asociada al encabezado de la fila.
        # 
        # Ejemplo en este contexto:
        # En el ejemplo del campo "President", después del <th> encontramos:
        # <td class="infobox-data">
        #     <a href="/w/index.php?title=Geraldine_Rauch&amp;action=edit&amp;redlink=1" class="new" title="Geraldine Rauch (page does not exist)">
        #         Geraldine Rauch
        #     </a>
        #     <small>(since 2022)</small>
        # </td>
        data_cell = row.find_next_sibling("td")
        
        
        if data_cell:
            # Usamos `get_text(strip=True)` para obtener el contenido textual de la celda <td>.
            # `get_text(strip=True)` nos permite:
            # - Extrae todo el texto de la celda, ignorando etiquetas HTML internas como <a> o <small>.
            # - Elimina espacios adicionales al inicio y al final del texto.
            # 
            # Ejemplo en este contexto:
            # Para el contenido de la celda:
            # <td class="infobox-data">
            #     <a href="/w/index.php?title=Geraldine_Rauch&amp;action=edit&amp;redlink=1" class="new" title="Geraldine Rauch (page does not exist)">
            #         Geraldine Rauch
            #     </a>
            #     <small>(since 2022)</small>
            # </td>
            # `get_text(strip=True)` devolverá: "Geraldine Rauch (since 2022)".
            return data_cell.get_text(strip=True)
            
    # Si no encontramos la celda <th> o no hay una celda <td> asociada, devolvemos "No applicable".
    return "No applicable"

In [35]:
def extract_president_or_rector(table):
    """
    Extrae el valor del campo 'President' o 'Rector', devolviendo el que esté disponible. Esto lo hacemos ya que el nombre oficial de la
    figura cambia de una universidad a otra. Por ejemplo, para la Technische Universität Berlin la figura oficial es President mientras que
    para University of Augsburg, la figura es Rector.

    Input:
        table (Tag): La tabla HTML que contiene los datos.

    Output:
        str: El valor del campo 'President' o 'Rector' o "No applicable" si ninguno está disponible.
    """
    # Buscamos el valor del campo 'President' en la tabla.
    president = extract_field_value(table, "President")
    # Si encontramos un valor para 'President', lo devolvemos.
    if president != "No applicable":
        return president
    # Si no, buscamos el campo 'Rector'.
    rector = extract_field_value(table, "Rector")
    # Devolvemos el valor de 'Rector' o "No applicable" si tampoco está disponible.
    return rector if rector != "No applicable" else "No applicable"

In [37]:
def extract_established_or_active(table):
    """
    Extrae el valor del campo 'Established' o 'Active', devolviendo el que esté disponible. Este campo contiene información sobre cuando
    fue fundada la universidad. De manera similar a lo que sucede con Rector, para algunas universidades esta información aparece como
    Established (e.g. University of Augsburg)y en otras como Active (e.g. Technical University of Kaiserslautern).

    Input:
        table (Tag): La tabla HTML que contiene los datos.

    Ouput:
        str: El valor del campo 'Established' o 'Active' o "No applicable" si ninguno está disponible.
    """
    # Buscamos el valor del campo 'Established' en la tabla.
    established = extract_field_value(table, "Established")
    
    if established != "No applicable":
        return established
        
    # Si no está 'Established', buscamos el campo 'Active'.
    active = extract_field_value(table, "Active")
    
    return active if active != "No applicable" else "No applicable"

In [39]:
def extract_budget_or_endowment(table):
    """
    Extrae el valor del campo 'Budget' o 'Endowment', devolviendo el que esté disponible. 
    Esto es útil porque en algunas universidades, el campo relacionado con el presupuesto puede
    aparecer bajo dos nombres distintos: 'Budget' o 'Endowment', al igual que sucedía con 

    Input:
        table (Tag): La tabla HTML que contiene los datos (la infobox).

    Output:
        str: El valor del campo 'Budget' o 'Endowment', o "No applicable" si ninguno está disponible.

    Ejemplo de HTML:
    - Caso 'Budget':
        <tr>
            <th scope="row" class="infobox-label">Budget</th>
            <td class="infobox-data">€648.8 million</td>
        </tr>
    - Caso 'Endowment':
        <tr>
            <th scope="row" class="infobox-label">Endowment</th>
            <td class="infobox-data">approx. €38 million</td>
        </tr>
    """
    
    # Intentamos extraer el valor del campo 'Budget' usando la función `extract_field_value`.
    # Esta función busca en la tabla la celda <th> con el texto "Budget" y extrae el valor de la celda <td> asociada.
    # 
    # Ejemplo de uso:
    # Si el HTML contiene:
    # <th>Budget</th>
    # <td>€648.8 million</td>
    # El valor de `budget` será: "€648.8 million".
    # 
    # Si el campo 'Budget' no está presente en la tabla, la función devuelve "No applicable".
    budget = extract_field_value(table, "Budget")
    
    # Verificamos si el valor de 'Budget' es válido (es decir, distinto de "No applicable").
    # 
    # Si encontramos un valor para 'Budget', lo devolvemos directamente.
    # Ejemplo:
    # - Si `budget = "€648.8 million"`, devolvemos ese valor.
    if budget != "No applicable":
        return budget

    # Si no encontramos 'Budget', intentamos buscar el valor del campo 'Endowment'.
    # 
    # Ejemplo de uso:
    # Si el HTML contiene:
    # <th>Endowment</th>
    # <td>approx. €38 million</td>
    # El valor de `endowment` será: "approx. €38 million".
    # 
    # Si el campo 'Endowment' no está presente en la tabla, la función devuelve "No applicable".
    endowment = extract_field_value(table, "Endowment")

    # Comprobamos si encontramos un valor para 'Endowment'.
    # Si 'Endowment' no está disponible, devolvemos "No applicable" como valor predeterminado.
    # - Si `endowment != "No applicable"`, devolvemos el valor de `endowment`.
    # - En caso contrario, devolvemos "No applicable".
    return endowment if endowment != "No applicable" else "No applicable"

In [41]:
def extract_students_or_postgraduates(table):
    """
    Extrae el valor del campo 'Students' o 'Postgraduates', devolviendo el que esté disponible.
    En algunas universidades, el número total de estudiantes aparece bajo el nombre 'Students',
    mientras que en otras puede aparecer como 'Postgraduates' (solo estudiantes de posgrado).

    Input:
        table (Tag): La tabla HTML que contiene los datos (la infobox).

    Output:
        str: El valor del campo 'Students' o 'Postgraduates', o "No applicable" si ninguno está disponible.
    """
    
    # Intentamos extraer el valor del campo 'Students' de la tabla.
    # 
    # `extract_field_value(table, "Students"):
    # Busca en la tabla una celda <th> con el texto "Students" y extrae el contenido de la celda <td> asociada.
    # 
    # Ejemplo de HTML:
    # <tr>
    #     <th>Students</th>
    #     <td>14,869<sup id="cite_ref-auto1_4-2" class="reference">
    #         <a href="#cite_note-auto1-4"><span class="cite-bracket">[</span>4<span class="cite-bracket">]</span></a>
    #     </sup></td>
    # </tr>
    # Resultado esperado: students = "14,869"
    students = extract_field_value(table, "Students")
    
    # Verificamos si encontramos un valor válido para 'Students'.
    # 
    # ¿Qué es un valor válido?
    # Cualquier valor que no sea "No applicable".
    # 
    # Ejemplo de flujo:
    # - Si `students = "14,869"`, devolvemos directamente ese valor.
    # - Si `students = "No applicable"`, continuamos buscando el campo 'Postgraduates'.
    if students != "No applicable":
        return students

    # Si no encontramos 'Students', buscamos el valor del campo 'Postgraduates'.
    # 
    # Ejemplo de HTML (simplificado):
    # <tr>
    #     <th>Postgraduates</th>
    #     <td>Approx. 400</td>
    # </tr>
    # Resultado esperado: postgraduates = "Approx. 400"
    postgraduates = extract_field_value(table, "Postgraduates")

    # Verificamos si encontramos un valor válido para 'Postgraduates'.
    # 
    # Si encontramos un valor, lo devolvemos.
    # Si no encontramos nada, devolvemos "No applicable".
    # 
    # Ejemplo de flujo:
    # - Si `postgraduates = "Approx. 400"`, devolvemos ese valor.
    # - Si `postgraduates = "No applicable"`, devolvemos "No applicable".
    return postgraduates if postgraduates != "No applicable" else "No applicable"

In [43]:
def extract_infobox_data(html_content, file_name):
    """
    Extrae los campos específicos de la infobox de una universidad desde el contenido HTML.

    Input:
        html_content (str): Contenido HTML del archivo de la universidad.
        file_name (str): Nombre del archivo que contiene el HTML procesado.

    Output:
        dict: Un diccionario con los datos extraídos de la infobox, o valores predeterminados
              si no se encuentra la tabla.

    """

    # Analizamos el contenido HTML con BeautifulSoup.
    # 
    # ¿Qué hace esta línea?
    # Convierte el contenido HTML en un objeto que BeautifulSoup puede analizar.
    # Esto nos permite buscar y extraer elementos específicos del HTML como tablas, filas, celdas, etc.
    # 
    soup = BeautifulSoup(html_content, "html.parser")

    # Buscamos la tabla que contiene la infobox.
    # Para ello, buscamos una tabla HTML con la clase "infobox vcard", que es el formato estándar de las infoboxes en Wikipedia.
    # 
    # Ejemplo de uso:
    # Si `html_content` contiene:
    # <table class="infobox vcard">
    #     <caption>Technical University of Munich</caption>
    # </table>
    # La variable `table` contendrá este bloque HTML.
    table = soup.find("table", class_="infobox vcard")

    # Si no encontramos la tabla, devolvemos un diccionario con valores predeterminados.
    # Esto asegura que siempre devolvemos un resultado, incluso si el HTML no contiene la tabla esperada.
    # 
    # - Si `table` es None, devolvemos un diccionario con "No applicable" en todos los campos.
    if not table:
        return {
            "file_name": file_name, #Lo añadimos ya que nos hace más fácil localizar el html cuando encontremos algún error en el csv.
            "name": "No applicable",
            "type": "No applicable",
            "established/active": "No applicable",
            "budget/endowment": "No applicable",
            "president/rector": "No applicable",
            "students/postgraduates": "No applicable",
            "website": "No applicable",
        }

    # Extraemos el nombre de la universidad de la etiqueta <caption>.
    # 
    # ¿Qué hace esta línea?
    # Busca un elemento <caption> con la clase "infobox-title fn org", que suele contener el nombre de la universidad.
    # 
    # Ejemplo:
    # Para este HTML:
    # <caption class="infobox-title fn org">Technical University of Munich</caption>
    # La variable `name` contendrá el texto: "Technical University of Munich".
    name = table.find("caption", class_="infobox-title fn org")

    # Creamos un diccionario con los datos extraídos.
    # 
    # 
    # Cada clave representa un campo de la infobox (por ejemplo, "type" o "website").
    # Cada valor se obtiene llamando a una función específica que extrae ese dato de la tabla.
    # 
    # Ejemplo de contenido final:
    # {
    #     "file_name": "tum.html",
    #     "name": "Technical University of Munich",
    #     "type": "Public",
    #     ...
    # }
    fields = {
        # Guardamos el nombre del archivo actual.
        "file_name": file_name,

        # Guardamos el nombre de la universidad, o "No applicable" si no se encuentra.
        # 
        # Ejemplo:
        # Si `name` es None, el resultado será: "No applicable".
        # Si `name` contiene "Technical University of Munich", el resultado será ese texto.
        "name": name.get_text(strip=True) if name else "No applicable",

        # Extraemos el tipo de universidad (pública o privada).
        "type": extract_field_value(table, "Type"),

        # Extraemos el año de fundación o el estado activo de la universidad.
        "established/active": extract_established_or_active(table),

        # Extraemos el presupuesto o la dotación de fondos.
        "budget/endowment": extract_budget_or_endowment(table),

        # Extraemos el presidente o rector.
        "president/rector": extract_president_or_rector(table),

        # Extraemos el número de estudiantes o postgraduados.
        "students/postgraduates": extract_students_or_postgraduates(table),

        # Extraemos el sitio web de la universidad.
        "website": extract_field_value(table, "Website"),
    }

    # Devolvemos el diccionario con los datos extraídos.
    return fields

In [45]:
def crawl_html_files(input_dir, output_csv):
    """
    Procesa los archivos HTML en un directorio y genera un archivo CSV con los datos extraídos.

    Input:
        input_dir (str): Directorio que contiene los archivos HTML.
        output_csv (str): Ruta del archivo CSV donde se guardarán los datos extraídos.

    Output:
        None

    Ejemplo de uso:
        - input_dir = "university_htmls"
        - output_csv = "universities_data.csv"
    """
    # Inicializamos una lista vacía donde almacenaremos los datos extraídos de cada archivo HTML.
    #
    # Esto nos permitirá recopilar los datos de todas las universidades antes de escribirlos al archivo CSV.
    all_data = []

    # Recorremos todos los archivos en el directorio especificado por `input_dir`.
    #
    # Usamos `os.listdir` para obtener la lista de todos los nombres de archivo en el directorio.
    # Luego, procesamos cada archivo por separado en el bucle.
    for filename in os.listdir(input_dir):

        # Filtramos los archivos para asegurarnos de que solo procesamos los que tienen extensión ".html". Esto lo hacemos por seguridad,
        # aunque sabemos que en nuestra carpeta, que creamos en la práctica anterior, solo hay archivos html.
        # 
        # Ejemplo:
        # - Si `filename = "university1.html"`, pasamos al siguiente paso.
        # - Si `filename = "notes.txt"`, lo ignoramos.
        if filename.endswith(".html"):

            # Construimos la ruta completa del archivo.
            # 
            # 
            # Ejemplo:
            # - Si `input_dir = "university_htmls"` y `filename = "tum.html"`,
            #   entonces `filepath = "university_htmls/tum.html"`.
            filepath = os.path.join(input_dir, filename)

            # Abrimos el archivo HTML en modo lectura (`"r"`) y con codificación UTF-8.
            # 
            # UTF-8 nos permite procesar correctamente caracteres especiales que puedan estar presentes en el HTML.
            # 
            # Ejemplo:
            # - Si el archivo contiene el texto "Kühne Logistics University ", la codificación UTF-8 lo procesa correctamente.
            with open(filepath, "r", encoding="utf-8") as file:

                # Leemos el contenido completo del archivo HTML.
                # 
                # Almacenamos el contenido HTML del archivo  en la variable `html_content`.
                # 
                # Ejemplo:
                # Si el archivo contiene:
                # <html><body><table class="infobox vcard">...</table></body></html>
                # Entonces `html_content` guardará ese contenido.
                html_content = file.read()

                # Extraemos los datos específicos de la infobox usando la función `extract_infobox_data`.
                # 
                # Llama a `extract_infobox_data`, que analiza el contenido del HTML y extrae los datos
                # relevantes de la tabla con clase "infobox vcard".
                # 
                # Ejemplo:
                # Si `filename = "university1.html"` y el HTML contiene datos válidos,
                # entonces `data` será un diccionario como:
                # {
                #     "file_name": "university1.html",
                #     "name": "University of Example",
                #     "type": "Public",
                #     ...
                # }
                data = extract_infobox_data(html_content, filename)

                # Filtramos los casos en los que no se encontró un nombre válido.
                # 
                # Si el campo `name` contiene "No applicable", significa que no encontramos datos útiles
                # en el archivo HTML, por lo que no lo incluimos en el CSV final. Esto puede deberse a que la página de la Wikipedia de
                # la universidad no contenga una infobox de la que extraer los datos.
                if data["name"] != "No applicable":

                    # Agregamos el diccionario `data` a la lista `all_data`.
                    # 
                    # Ejemplo:
                    # - Si `data = {"file_name": "tum.html", "name": "Technical University of Munich", ...}`,
                    #   entonces `all_data` ahora contiene:
                    #   [{"file_name": "tum.html", "name": "Technical University of Munich", ...}]
                    all_data.append(data)

    # Especificamos los nombres de las columnas que tendrá el archivo CSV.
    fieldnames = [
        "file_name", "name", "type", "established/active", "budget/endowment",
        "president/rector", "students/postgraduates", "website"
    ]

    # Abrimos el archivo CSV en modo escritura (`"w"`) y con codificación UTF-8.
    #
    # Preparamos el archivo especificado por `output_csv` para que podamos escribir los datos en él.
    # Usamos `newline=""` para evitar líneas en blanco adicionales en el CSV.
    with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:

        # Creamos un objeto `DictWriter` para escribir los datos al archivo CSV.
        # 
        # `csv.DictWriter`. Nos permite escribir diccionarios directamente al archivo CSV, usando las claves como nombres de columnas.
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # Escribimos la fila de encabezados en el archivo CSV.
        # 
        # La primera fila del archivo CSV será:
        # "file_name","name","type","established/active","budget/endowment",...
        writer.writeheader()

        # Escribimos todas las filas de datos al archivo CSV.
        # 
        # `writer.writerows`? Toma la lista `all_data` (que contiene diccionarios) y escribe cada diccionario como una fila en el CSV.
        # 
        writer.writerows(all_data)

In [47]:
if __name__ == "__main__":
    """
    Este bloque garantiza que el código dentro de él solo se ejecutará si el script
    se ejecuta directamente como programa principal y no cuando se importe como módulo en otro script.

    Ejemplo:
    - Si ejecutamos este archivo directamente con `python crawler.py`, este bloque se ejecutará.
    - Si importamos funciones desde este archivo en otro script con `import crawler`, el bloque NO se ejecutará.

    Propósito:
    - Este bloque inicializa las variables necesarias y llama a la función principal (`crawl_html_files`).
    """

    # Ruta al directorio donde se encuentran los archivos HTML.
    # Aquí asumimos que los archivos HTML están en una carpeta llamada "university_htmls".
    input_dir = "university_htmls"

    # Nombre del archivo CSV donde se guardarán los datos procesados.
    # Esto es útil para consolidar los datos en un formato fácilmente reutilizable.
    output_csv = "universities_data.csv"

    # Llamamos a la función principal para procesar los archivos HTML y generar el CSV.
    # Se pasan las rutas definidas anteriormente como argumentos.
    crawl_html_files(input_dir, output_csv)